In [2]:
%%writefile CRUD.py

from fastapi import FastAPI
from pydantic import BaseModel
from sqlalchemy import create_engine, Integer, Column, String, Boolean
from sqlalchemy.orm import declarative_base, sessionmaker

app = FastAPI()

DATABASE_URL = "sqlite:///./todo.db"

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

Base = declarative_base()

SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)


class TaskDB(Base):
    __tablename__ = "tasks"

    id = Column(Integer, primary_key=True)
    title = Column(String)
    is_done = Column(Boolean, default=False)


Base.metadata.create_all(bind=engine)


class Task(BaseModel):
    id: int
    title: str
    is_done: bool = False


@app.post("/tasks")
def create_task(task: Task):
    db = SessionLocal()

    new_task = TaskDB(
        id=task.id,
        title=task.title,
        is_done=task.is_done
    )

    db.add(new_task)
    db.commit()
    db.refresh(new_task)
    db.close()

    return new_task


@app.get("/tasks")
def get_task():
    db = SessionLocal()

    tasks = db.query(TaskDB).all()

    db.close()

    return tasks

Writing CRUD.py
